# Who Stole my lunch?

The goal of this notebook is to solve, using a score-based generative model, the following problem:

Somebody stole the lunch of one of the employes of a given company. The employe got access to a security camera image of somebody exiting the common area with his lunch but the picture is low quality. Who stole his lunch?

In [44]:
#!pip install accelerate diffusers facenet-pytorch datasets scikit-learn

In [ ]:
from diffusers import DDIMScheduler, UNet2DModel
from PIL import Image
import torch
from torchvision.transforms.functional import pil_to_tensor
from torchvision.transforms import Compose, Normalize
from datasets import load_dataset
from facenet_pytorch import MTCNN, InceptionResnetV1
from tqdm import tqdm
import numpy as np


## Helper functions
def img_to_tensor(img):
    return (pil_to_tensor(img) / 255 - 0.5)*2

def tensors_to_img(dt, rescale=True):
    if rescale:
        images = ((dt / 2 + 0.5).clamp(0, 1)*255).round()
    else:
        images = dt
    images = images.cpu().permute(0, 2, 3, 1).numpy()
    images = [Image.fromarray(image.astype(np.uint8)) for image in images]
    return images

def image_grid(imgs, rows, cols):
    assert len(imgs) == rows * cols

    w, h = imgs[0].size
    grid = Image.new("RGB", size=(cols * w, rows * h))
    grid_w, grid_h = grid.size

    for i, img in enumerate(imgs):
        grid.paste(img, box=(i % cols * w, i // cols * h))
    return grid


## Loading dataset
ds = load_dataset("korexyz/celeba-hq-256x256")

#Loading diffusion related objects
model_id = "google/ddpm-celebahq-256"
scheduler = DDIMScheduler.from_pretrained(model_id)
model = UNet2DModel.from_pretrained(model_id).to("cuda").eval().requires_grad_(False)

# Setting number of steps for the reverse diffusion.
scheduler.set_timesteps(50)


# Face detection pipeline MTCNN:
mtcnn = MTCNN(keep_all=True, select_largest=True, post_process=True, device="cuda:0").requires_grad_(False) # Receives PIL images as inputs!

# Embedding network: For the picture of a face (output of mtcnn) create an embedding vector that represents the face.
#This embedding is trained using contrastive learning so that pictures of the same person are close together than pictures of two different people.
resnet = InceptionResnetV1(pretrained='vggface2').eval().to("cuda:0").requires_grad_(False)



In [3]:
# Indexes of the validation set corresponding to the "suspects" and the observation
observation_index = 499
suspect_indexes = [1559, 155, 1979, 818, 1313, 1870, 1821, 846, 1485, 1844, 548, 1564, 521, 621, 321, 1316, 220, 650, 1981, 598]

# Operator that goes from high resolution to low resolution. Equivalent of the forward operator.
low_res_operator = torch.nn.AvgPool2d(18, 18)


#This might take a while to run...
observation = low_res_operator(img_to_tensor(ds["validation"]["image"][observation_index]))
observation_img = tensors_to_img(observation[None])[0]

suspect_images = [ds["validation"]["image"][idx] for idx in suspect_indexes]

## The photo of the suspect

In [ ]:
observation_img.resize((256, 256), Image.Resampling.LANCZOS)

## All your coworkers

In [ ]:
image_grid(suspect_images, rows=2, cols=10)

### Suggestions to begin with:

1. It is a good idea to start by implementing just the generative model part. This can help you: https://huggingface.co/docs/diffusers/using-diffusers/write_own_pipeline
2. Implement then the DPS algorithm to sample from the inverse problem: https://arxiv.org/abs/2209.14687
